In [1]:
library(tidyverse)
library(RadioGx)
library(dplyr)
library(purrr)
library(readr)
library(stringr)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: CoreGx

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:lubridate’:

    as.difftime


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.

In [2]:
pSet <- readRDS("../data/Cleveland.rds")

# Loading RNASeq data
expr_matrix <- molecularProfiles(pSet, "rnaseq")

# Using the clean 'EnsemblGeneId' column directly from featureInfo output
gene_annotation <- as.data.frame(featureInfo(pSet, "rnaseq")) %>% 
  select(EnsemblGeneId, Symbol)

In [3]:
# Loading master TSV file
master_tsv <- read_tsv("../output/literature_search/Master_Tier1_Stress_Targets.tsv")

# Seperate by Stress_Type
stress_groups <- unique(master_tsv$Stress_Type)

# Initialize a list to store just the top 3 PCs for each pathway
pca_components_list <- list()

Rows: 2144 Columns: 6
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): Protein_Names, Gene_Names, Gene_Ontology, Stress_Type
dbl (2): Supporting_PMID, Paper_Count

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
for (stress in stress_groups) {
  message("Extracting PCs for: ", stress)
  
  # Get the clean, individual gene symbols for this pathway
  pathway_symbols <- master_tsv %>% 
    filter(Stress_Type == stress) %>% 
    mutate(clean_gene = str_split_i(Gene_Names, " ", 1)) %>% 
    pull(clean_gene)
  
  # Translate Gene Symbols to Ensembl IDs
  matching_ensembl_ids <- gene_annotation %>% 
    filter(Symbol %in% pathway_symbols) %>% 
    pull(EnsemblGeneId)
  
  # Double-check which of these Ensembl IDs actually exist in the matrix rows
  final_genes <- intersect(matching_ensembl_ids, rownames(expr_matrix))
  
  if(length(final_genes) < 3) {
    warning("Not enough matching Ensembl IDs found for: ", stress)
    next
  }
  
  # Subset the matrix using the translated Ensembl IDs
  subset_matrix <- expr_matrix[final_genes, ]
  
  # Calculate variance for each gene and drop any that are completely constant
  gene_variances <- apply(subset_matrix, 1, var, na.rm = TRUE)
  clean_subset_matrix <- subset_matrix[gene_variances > 0, , drop = FALSE]
  
  # Make sure there is still have enough genes after removing
  if(nrow(clean_subset_matrix) < 3) {
    warning("Skipping ", stress, " — too many genes had zero variance.")
    next
  }
  
  # Transpose for PCA (Samples as rows, clean Ensembl IDs as columns)
  pca_input <- t(clean_subset_matrix)
  
  # Perform PCA safely now
  pca_output <- prcomp(pca_input, center = TRUE, scale. = TRUE)
  
  # Extract the top 3 PCs
  three_pcs <- as.data.frame(pca_output$x) %>% 
    rownames_to_column(var = "cellid") %>% 
    select(cellid, PC1, PC2, PC3)
  
  # Store the dataframe in our results list
  pca_components_list[[stress]] <- three_pcs
}

Extracting PCs for: Cell Death Regulators

Extracting PCs for: Contact Site Stress

Extracting PCs for: Genotoxic Stress

Extracting PCs for: Immune Stress

Extracting PCs for: Mechanical Stress

Extracting PCs for: Nutrient Stress

Extracting PCs for: Organelle Stress

Extracting PCs for: Proteasomal Stress

Extracting PCs for: Proteotoxic Stress

Extracting PCs for: Ribotoxic Stress



In [5]:
# Pure PCs
head(pca_components_list[[1]])

# Pull the raw sensitivity data
raw_sensitivity <- pSet@sensitivity$profiles
sensitivity_data <- as.data.frame(raw_sensitivity) %>% 
  rownames_to_column(var = "cellid") %>% 
  select(cellid, matches("SF2"))

,cellid,PC1,PC2,PC3
,<chr>,<dbl>,<dbl>,<dbl>
1,G20461.HSC-3.2,3.3025122,-1.5269463,-5.8405696
2,G20463.C2BBe1.2,-4.5548482,-2.8926210,-1.4645205
3,G20466.5637.2,0.6398966,-1.9082326,-5.5672969
4,G20468.DMS_53.2,-3.3899787,-0.7861565,2.8055354
5,G20469.JHOS-2.2,2.3334978,-0.8822956,0.2770924
6,G20471.Daoy.2,-2.8925839,3.5424599,-5.1576663


In [6]:
# Clean the sensitivity and metadata
metadata_df <- as.data.frame(phenoInfo(pSet, "rnaseq")) %>% 
  rownames_to_column(var = "rna_id") %>% 
  select(rna_id, cellid) 

raw_sensitivity <- pSet@sensitivity$profiles
sensitivity_clean <- as.data.frame(raw_sensitivity) %>% 
  rownames_to_column(var = "sens_id") %>% 
  mutate(cellid = str_split_i(sens_id, "_", 1)) %>% 
  select(cellid, SF2)

# Loop through every stress pathway
for (stress in names(pca_components_list)) {
  
  message("Processing final deployment assets for: ", stress)
  
  # Grab current PCA dataframe
  pca_df <- pca_components_list[[stress]]
  colnames(pca_df)[1] <- "rna_id"
  
  # Complete the relational merge
  visual_df <- pca_df %>% 
    inner_join(metadata_df, by = "rna_id") %>% 
    inner_join(sensitivity_clean, by = "cellid", relationship = "many-to-many") %>% 
    drop_na(SF2)
  
  # Collapse many-to-many replicates down to pure averages
  clean_model_df <- visual_df %>% 
    group_by(cellid) %>% 
    summarise(
      PC1 = mean(PC1, na.rm = TRUE),
      PC2 = mean(PC2, na.rm = TRUE),
      PC3 = mean(PC3, na.rm = TRUE),
      SF2 = mean(SF2, na.rm = TRUE),
      .groups = "drop"
    )
  
  # Compute correlation statistic
  cor_test_pc1 <- cor.test(clean_model_df$PC1, clean_model_df$SF2)
  stat_summary <- data.frame(
    Pathway = stress,
    PC1_Correlation_r = cor_test_pc1$estimate,
    p_value = cor_test_pc1$p.value
  )
  
  # Sanitized filename string (replacing spaces/slashes to be filesystem safe)
  safe_filename <- gsub("[ /]", "_", stress)
  
  # Export Spreadsheets (CSVs)
  write_csv(clean_model_df, paste0("../output/PCA/", safe_filename, "_clean_averages_for_modeling.csv"))
  write_csv(stat_summary, paste0("../output/PCA/", safe_filename, "_correlation_stats.csv"))
  write_csv(pca_components_list[[stress]], paste0("../output/PCA/", safe_filename, "_pure_PCA_coordinates.csv"))
  
  # Generate and save the visualization
  clean_plot <- ggplot(clean_model_df, aes(x = PC1, y = PC2, color = SF2)) +
    geom_point(size = 3.5, alpha = 0.8) +
    scale_color_viridis_c(option = "plasma") + 
    theme_minimal() +
    labs(
      title = paste("PCA Space Colored by Radiation Response:", stress),
      subtitle = "Averaged biological replicates; color gradient shows radiation survival (SF2)",
      x = "Principal Component 1 (PC1)", 
      y = "Principal Component 2 (PC2)",
      color = "SF2 Value"
    )
  
  ggsave(
    filename = paste0("../output/PCA/", safe_filename, "_PCA_radiation_plot.png"), 
    plot = clean_plot, 
    width = 8, 
    height = 6, 
    dpi = 300
  )
}

Processing final deployment assets for: Cell Death Regulators

Processing final deployment assets for: Contact Site Stress

Processing final deployment assets for: Genotoxic Stress

Processing final deployment assets for: Immune Stress

Processing final deployment assets for: Mechanical Stress

Processing final deployment assets for: Nutrient Stress

Processing final deployment assets for: Organelle Stress

Processing final deployment assets for: Proteasomal Stress

Processing final deployment assets for: Proteotoxic Stress

Processing final deployment assets for: Ribotoxic Stress



In [7]:
base_dir <- "../output/PCA/"
pathways <- c(
  "Organelle_Stress", "Proteotoxic_Stress", "Genotoxic_Stress", 
  "Nutrient_Stress", "Proteasomal_Stress", "Mechanical_Stress", 
  "Contact_Site_Stress", "Ribotoxic_Stress", "Immune_Stress", 
  "Cell_Death_Regulators"
)

# Read and process each file dynamically
merged_data <- pathways %>%
  map(function(pathway) {
    # Construct the exact file path
    file_path <- paste0(base_dir, pathway, "_clean_averages_for_modeling.csv")
    
    # Read the file if it exists
    if (file.exists(file_path)) {
      df <- read_csv(file_path, show_col_types = FALSE)
      
      # Rename PC columns to include the pathway prefix, but keep cellid and SF2 clean
      df <- df %>%
        rename_with(~ paste0(pathway, "_", .x), starts_with("PC"))
      
      return(df)
    } else {
      warning(paste("File not found:", file_path))
      return(NULL)
    }
  }) %>%
  # Remove any NULL entries if a file was missing
  compact()

In [8]:
# Join all tables side-by-side using 'cellid' and 'SF2' as the anchors
merged_matrix <- merged_data %>%
  purrr::reduce(inner_join, by = c("cellid", "SF2"))

# Create the binary classification column for ShinyFeatures
final_master_matrix <- merged_matrix %>%
  mutate(
    Radiosensitive = if_else(SF2 < 0.5, 1, 0)
  ) %>%
  # Relocate identifiers and the target column to the front for easy viewing
  relocate(cellid, SF2, Radiosensitive)

# Save the final master file
output_file <- paste0(base_dir, "master_shinyfeatures_input.csv")
write_csv(final_master_matrix, output_file)